# Extreme Performance Mock — Blind Read

**Simulation:** You are handed messy, experimental PyTorch code. In 45 minutes,
without editing or using AI, explain how to speed it up and what can break.
The rest of the repository is deliberately large, but this notebook is your
entrypoint. No official employer questions are included.

Identify control flow, computational hotspots, sync points, memory pressure,
causal correctness assumptions, and how to verify each proposed change.


## File: `perfkit/infra/config.py`


In [ ]:
"""Validation of researcher-controlled experiment configurations.

Defaults should be small enough to run with CPU-only PyTorch. The configuration
object deliberately permits several independent attention backends for
controlled experiments; training code is responsible for selecting one.
"""
from __future__ import annotations
from dataclasses import dataclass, asdict
import json

@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int = 384
    hidden_size: int = 96
    num_heads: int = 4
    num_layers: int = 2
    intermediate_size: int = 384
    max_position_embeddings: int = 256
    dropout: float = 0.0
    rms_norm_eps: float = 1e-5
    activation: str = 'gelu'
    tie_embeddings: bool = True
    positional: str = 'learned'

    def validate(self):
        if self.vocab_size < 8: raise ValueError('vocab_size < 8')
        if self.hidden_size <= 0 or self.hidden_size % self.num_heads:
            raise ValueError('head dimension must be integral')
        if self.num_layers < 1: raise ValueError('num_layers must be positive')
        if self.max_position_embeddings < 1: raise ValueError('invalid maximum sequence length')
        if not 0 <= self.dropout < 1: raise ValueError('dropout outside [0, 1)')
        if self.activation not in ('gelu', 'silu', 'relu'): raise ValueError('unknown activation')
        if self.positional not in ('learned', 'rope'): raise ValueError('unknown positional mode')
        return self

    @classmethod
    def from_dict(cls, obj):
        return cls(**obj).validate()

    @classmethod
    def from_json(cls, text):
        return cls.from_dict(json.loads(text))

    def as_json(self):
        return json.dumps(asdict(self), sort_keys=True)

@dataclass(frozen=True)
class TrainConfig:
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    micro_batch: int = 4
    accumulation: int = 4
    max_norm: float = 1.0
    label_smoothing: float = 0.0
    mixed_precision: str = 'none'
    diagnostics: bool = True

    def validate(self):
        if self.learning_rate <= 0: raise ValueError('learning rate')
        if self.micro_batch < 1 or self.accumulation < 1: raise ValueError('batch')
        if self.max_norm < 0: raise ValueError('max_norm')
        if not 0 <= self.label_smoothing < 1: raise ValueError('smoothing')
        if self.mixed_precision not in ('none','bf16','fp16'):
            raise ValueError('precision')
        return self

@dataclass(frozen=True)
class EvalConfig:
    val_batch_size: int = 8
    max_new_tokens: int = 24
    top_k: int = 0
    temperature: float = 1.0
    def validate(self):
        if self.val_batch_size < 1 or self.max_new_tokens < 0: raise ValueError('size')
        if self.top_k < 0 or self.temperature <= 0: raise ValueError('sampling')
        return self

@dataclass(frozen=True)
class WorkloadConfig:
    seed: int = 42
    train_count: int = 256
    val_count: int = 32
    min_length: int = 16
    max_length: int = 64
    vocab_size: int = 384
    skew: float = 0.0
    def validate(self):
        if min(self.train_count,self.val_count) < 0: raise ValueError('count')
        if not 2 <= self.min_length <= self.max_length: raise ValueError('length')
        if self.vocab_size < 8 or not 0 <= self.skew <= 1: raise ValueError('distribution')
        return self


## File: `perfkit/data/corpus.py`


In [ ]:
"""Synthetic traces with variable lengths, token skew, and split keys."""
from dataclasses import dataclass
import random

@dataclass(frozen=True)
class Example:
    sample_id: int
    user_id: int
    timestamp: int
    tokens: tuple[int, ...]
    fold: int

    @property
    def length(self): return len(self.tokens)


def synthetic_corpus(n, *, seed=17,vocab=384,min_length=16,max_length=64,
                     users=40, skew=0.0):
    if not 2 <= min_length <= max_length: raise ValueError('invalid lengths')
    if users < 1 or vocab < 8: raise ValueError('invalid users/vocab')
    rng=random.Random(seed)
    result=[]
    for i in range(n):
        length=rng.randint(min_length,max_length)
        if rng.random() < skew: length=max_length
        user=rng.randrange(users)
        tokens=[]
        for _ in range(length):
            tokens.append(1 if rng.random() < skew else rng.randrange(2,vocab))
        result.append(Example(i,user,i*7,tuple(tokens),user%5))
    return result


def split_by_time(examples,cutoff):
    train,val=[],[]
    for e in examples:
        (train if e.timestamp<cutoff else val).append(e)
    return train,val


def split_by_user(examples,holdout_users):
    holdout=set(holdout_users)
    train,val=[],[]
    for e in examples:
        (val if e.user_id in holdout else train).append(e)
    return train,val


def corpus_histogram(examples):
    from collections import Counter
    lengths=Counter(e.length for e in examples)
    tokens=Counter(t for e in examples for t in e.tokens)
    users=Counter(e.user_id for e in examples)
    return {'lengths':lengths,'tokens':tokens,'users':users}


def verify_no_overlap(train,val):
    a={e.sample_id for e in train}; b={e.sample_id for e in val}
    if a & b: raise ValueError('train/validation leakage')
    if len(a)!=len(train) or len(b)!=len(val):
        raise ValueError('duplicate sample id')
    return True


## File: `perfkit/data/batcher.py`


In [ ]:
"""Batch assembler; a realistic locus for CPU-to-device transfer costs."""
from dataclasses import dataclass
import torch

@dataclass
class TokenBatch:
    input_ids: torch.Tensor
    targets: torch.Tensor
    attention_mask: torch.Tensor
    lengths: torch.Tensor
    ids: tuple[int, ...]

    def to(self,device,non_blocking=False):
        return TokenBatch(self.input_ids.to(device,non_blocking=non_blocking),
                          self.targets.to(device,non_blocking=non_blocking),
                          self.attention_mask.to(device,non_blocking=non_blocking),
                          self.lengths.to(device,non_blocking=non_blocking),
                          self.ids)

    @property
    def useful_tokens(self): return int(self.attention_mask.sum().item())
    @property
    def padded_tokens(self): return int(self.input_ids.numel())


def pad_batch(records, *, pad=0, device='cpu', max_length=None):
    if not records: raise ValueError('empty batch')
    lengths=[len(e.tokens)-1 for e in records]
    if any(l<1 for l in lengths): raise ValueError('short sequence')
    width=min(max(lengths),max_length or max(lengths))
    xs=torch.full((len(records),width),pad,dtype=torch.long)
    ys=torch.full((len(records),width),-100,dtype=torch.long)
    mask=torch.zeros((len(records),width),dtype=torch.bool)
    for idx,example in enumerate(records):
        length=min(lengths[idx],width)
        xs[idx,:length]=torch.tensor(example.tokens[:length])
        ys[idx,:length]=torch.tensor(example.tokens[1:length+1])
        mask[idx,:length]=True
    return TokenBatch(xs.to(device),ys.to(device),mask.to(device),
                      torch.tensor(lengths).to(device),
                      tuple(e.sample_id for e in records))


def unpad_targets(batch):
    return [batch.targets[i,:int(batch.attention_mask[i].sum())].tolist()
            for i in range(batch.input_ids.shape[0])]


def make_microbatches(records, micro_batch, **kwargs):
    if micro_batch<1:raise ValueError('micro_batch')
    for start in range(0,len(records),micro_batch):
        yield pad_batch(records[start:start+micro_batch],**kwargs)


## File: `perfkit/model/attention.py`


In [ ]:
"""Research attention: two implementations with a shared projection state.

The reference backend intentionally materializes full attention scores.
The alternate backend is a comparison target, not necessarily a drop-in for
all masks, dropout settings, backends and dtypes.
"""
import math
import torch
from torch import nn
from torch.nn import functional as F
from .rope import apply_rotary

class ResearchMultiheadAttention(nn.Module):
    def __init__(self,width,heads,dropout=0.0,use_rope=False):
        super().__init__()
        if width%heads:raise ValueError('width/heads')
        self.heads=heads
        self.dim=width//heads
        self.qkv=nn.Linear(width,3*width)
        self.out=nn.Linear(width,width)
        self.dropout=nn.Dropout(dropout)
        self.use_rope=use_rope

    def project(self,x,offset=0):
        batch,seq,width=x.shape
        q,k,v=self.qkv(x).chunk(3,-1)
        def arrange(x):return x.reshape(batch,seq,self.heads,self.dim).transpose(1,2)
        q,k,v=map(arrange,(q,k,v))
        if self.use_rope:q,k=apply_rotary(q,k,offset=offset)
        return q,k,v

    def slow_scores(self,q,k):
        logits=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.dim)
        length=logits.shape[-1]
        block=torch.triu(torch.ones(length,length,device=logits.device,
                                     dtype=torch.bool),diagonal=1)
        return logits.masked_fill(block,float('-inf'))

    def forward(self,x,backend='research',offset=0):
        batch,seq,width=x.shape
        q,k,v=self.project(x,offset)
        if backend=='research':
            outputs=[]
            for head in range(self.heads):
                logit=self.slow_scores(q[:,head:head+1],k[:,head:head+1])
                probs=F.softmax(logit,dim=-1)
                outputs.append(self.dropout(probs)@v[:,head:head+1])
            y=torch.cat(outputs,dim=1)
        elif backend=='sdpa':
            y=F.scaled_dot_product_attention(q,k,v,dropout_p=
                 self.dropout.p if self.training else 0.0,is_causal=True)
        else:raise ValueError('unknown backend')
        return self.out(y.transpose(1,2).contiguous().reshape(batch,seq,width))


## File: `perfkit/model/blocks.py`


In [ ]:
"""Transformer layer assembly with intentionally unfused subgraphs."""
import torch
from torch import nn
from torch.nn import functional as F
from .attention import ResearchMultiheadAttention
from .norms import SlowRMSNorm

class FeedForward(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.up=nn.Linear(config.hidden_size,config.intermediate_size)
        self.down=nn.Linear(config.intermediate_size,config.hidden_size)
        self.activation=config.activation
        self.dropout=nn.Dropout(config.dropout)

    def forward(self,x):
        h=self.up(x)
        if self.activation=='gelu':h=F.gelu(h)
        elif self.activation=='silu':h=F.silu(h)
        else:h=F.relu(h)
        return self.dropout(self.down(h))

class TransformerBlock(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.pre_attn=SlowRMSNorm(config.hidden_size,config.rms_norm_eps)
        self.attn=ResearchMultiheadAttention(config.hidden_size,config.num_heads,
                 config.dropout,config.positional=='rope')
        self.pre_ffn=SlowRMSNorm(config.hidden_size,config.rms_norm_eps)
        self.ffn=FeedForward(config)
        self.drop=nn.Dropout(config.dropout)

    def forward(self,x,backend='research'):
        x=x+self.drop(self.attn(self.pre_attn(x),backend=backend))
        x=x+self.drop(self.ffn(self.pre_ffn(x)))
        return x


## File: `perfkit/model/decoder.py`


In [ ]:
"""End-to-end causal decoder and configurable embedding semantics."""
import torch
from torch import nn
from .blocks import TransformerBlock
from .norms import SlowRMSNorm

class ResearchDecoder(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config=config.validate()
        self.embed=nn.Embedding(config.vocab_size,config.hidden_size)
        self.positions=(nn.Embedding(config.max_position_embeddings,
                       config.hidden_size) if config.positional=='learned' else None)
        self.blocks=nn.ModuleList(TransformerBlock(config)
                                  for _ in range(config.num_layers))
        self.norm=SlowRMSNorm(config.hidden_size,config.rms_norm_eps)
        self.lm_head=nn.Linear(config.hidden_size,config.vocab_size,bias=False)
        if config.tie_embeddings:self.lm_head.weight=self.embed.weight

    def encode(self,ids,backend='research',positions_offset=0):
        if ids.ndim!=2:raise ValueError('ids rank')
        batch,length=ids.shape
        if length==0 or length+positions_offset>self.config.max_position_embeddings:
            raise ValueError('sequence too long')
        state=self.embed(ids)
        if self.positions is not None:
            pos=torch.arange(positions_offset,positions_offset+length,
                             device=ids.device)
            state=state+self.positions(pos)[None,:,:]
        for block in self.blocks:state=block(state,backend=backend)
        return self.norm(state)

    def forward(self,ids,backend='research'):
        return self.lm_head(self.encode(ids,backend=backend))

    def param_groups(self,weight_decay):
        decay=[]; no_decay=[]
        for name,p in self.named_parameters():
            if p.ndim>=2:decay.append(p)
            else:no_decay.append(p)
        return [{'params':decay,'weight_decay':weight_decay},
                {'params':no_decay,'weight_decay':0.0}]


## File: `perfkit/train/engine.py`


In [ ]:
"""Inefficient but readable experimental loop; optimize end-to-end, not locally.

Its diagnostics are intentionally expensive and can dominate GPU workloads.
Train semantics and reduction denominators remain correctness requirements.
"""
from dataclasses import dataclass
import time
import torch
from torch import nn
from .losses import causal_token_loss
from .optimizers import inspect_gradients,clip_gradients
from ..data.batcher import pad_batch

@dataclass
class TrainStep:
    loss: float
    examples: int
    tokens: int
    elapsed: float
    gradient_summary: dict

class ResearchTrainer:
    def __init__(self,model,optimizer,device='cpu',diagnostics=True,backend='research'):
        self.model=model
        self.optimizer=optimizer
        self.device=torch.device(device)
        self.diagnostics=diagnostics
        self.backend=backend
        self.steps=0

    def one_step(self,examples,microbatch=4,clip_norm=1.0):
        if not examples:raise ValueError('empty examples')
        self.model.train()
        self.optimizer.zero_grad(set_to_none=True)
        t0=time.perf_counter()
        chunks=[examples[x:x+microbatch] for x in range(0,len(examples),microbatch)]
        total=0.0
        token_count=0
        for chunk in chunks:
            batch=pad_batch(chunk,device=self.device,
                            max_length=self.model.config.max_position_embeddings)
            logits=self.model(batch.input_ids,backend=self.backend)
            loss=causal_token_loss(logits,batch.targets)/len(chunks)
            loss.backward()
            total+=float(loss.item())
            token_count+=batch.useful_tokens
        stats=inspect_gradients(self.model) if self.diagnostics else {}
        clip_gradients(self.model,clip_norm)
        self.optimizer.step()
        self.steps+=1
        if self.device.type=='cuda':torch.cuda.synchronize(self.device)
        return TrainStep(total,len(examples),token_count,
                         time.perf_counter()-t0,stats)

    def train_steps(self,examples,n_steps,batch_size,microbatch=4):
        results=[]
        for i in range(n_steps):
            start=(i*batch_size)%max(1,len(examples))
            batch=examples[start:start+batch_size]
            if not batch:batch=examples[:batch_size]
            results.append(self.one_step(batch,microbatch))
        return results


## File: `perfkit/eval/evaluator.py`


In [ ]:
"""Validation deliberately runs serially and frequently synchronizes GPU."""
from dataclasses import dataclass
import torch
from .metrics import token_nll_sum,summarize_nll
from ..data.batcher import pad_batch

@dataclass
class ValidationResult:
    mean_nll: float
    perplexity: float
    valid_tokens: int
    examples: int


def evaluate(model,examples,device='cpu',backend='research'):
    model.eval()
    dev=torch.device(device)
    total_nll=0.0
    total_count=0
    for example in examples:
        x=pad_batch([example],device=dev)
        logits=model(x.input_ids,backend=backend)
        loss=token_nll_sum(logits,x.targets)
        total_nll+=float(loss.item())
        total_count+=int(x.attention_mask.sum().item())
    metrics=summarize_nll(total_nll,total_count)
    return ValidationResult(metrics['mean_nll'],metrics['perplexity'],
                            total_count,len(examples))


def evaluate_inference_mode(model,examples,device='cpu',backend='research'):
    with torch.inference_mode():
        return evaluate(model,examples,device,backend)


## File: `perfkit/serve/decoding.py`


In [ ]:
"""Autoregressive decode calls the full transformer at every new token."""
import time
import torch
from .sampling import sample_next

@torch.no_grad()
def generate(model,prefix,new_tokens,*,temperature=1.0,top_k=0,
             deterministic=True,backend='research'):
    if prefix.ndim!=2 or prefix.shape[1]<1:raise ValueError('prefix')
    tokens=prefix
    for _ in range(new_tokens):
        context=tokens[:,-model.config.max_position_embeddings:]
        logits=model(context,backend=backend)
        token=sample_next(logits[:,-1],temperature,top_k,deterministic)
        tokens=torch.cat((tokens,token),dim=1)
    return tokens


def benchmark_decode(model,prefix,new_tokens,*,backend='research',repeats=3):
    times=[];out=None
    device=prefix.device
    for _ in range(repeats):
        if device.type=='cuda':torch.cuda.synchronize(device)
        start=time.perf_counter()
        out=generate(model,prefix,new_tokens,backend=backend)
        if device.type=='cuda':torch.cuda.synchronize(device)
        times.append(time.perf_counter()-start)
    return {'tokens_generated':prefix.shape[0]*new_tokens,
            'times':times,'last_token':out[:,-1]}


## Oral examination
State your top bottlenecks and justify evidence before you write code.
Then run profiling and test changes in the repository; do not rely on this
notebook alone as a regression suite.
